# 05 · LLM answer evaluation

Real answers from the deployed assistant (Gemini) for synthetic held-out questions. Failed or unavailable generations are counted, not dropped. Automatic, reference-free metrics are computed here; correctness, relevance and faithfulness require human review with the fixed rubric (`configs/answer_rubric_v1.md`) and are reported only when reviewed scores exist.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
f = A.RESULTS / "answers_production_heldout_v0.2.jsonl"
answers = pd.read_json(f, lines=True) if f.exists() else pd.DataFrame()
ann_f = A.DATA / "review" / "answer_annotation_template.csv"
annotations = pd.read_csv(ann_f) if ann_f.exists() else pd.DataFrame()
print(f"{len(answers)} recorded answer cases" if len(answers) else "Not evaluated: no answer-evaluation run found")

In [ ]:
if len(answers):
    meta = json.loads(Path(str(f) + ".manifest.json").read_text())
    display(pd.DataFrame([(k, str(v)) for k, v in meta.items()], columns=["run", "value"]))
    outcome = answers["answer_generation"].fillna("request failed: " + answers["request_error"].fillna("unknown"))
    display(outcome.value_counts().rename_axis("generation outcome").reset_index(name="cases"))
    display(answers.groupby("answer_generation", dropna=False)["fallback_reason"].value_counts(dropna=False).rename("cases").reset_index())

## Automatic metrics (generated answers only; n shown per metric)

In [ ]:
def rate(s):
    s = s.dropna().astype(float)
    return (s.mean() if len(s) else np.nan), len(s)
if len(answers):
    gen = answers[answers["answer_generation"] == "llm"]
    rows = [("generation success rate (all cases)", *rate((answers["answer_generation"] == "llm").astype(float)))]
    rows += [("language consistency", *rate(gen["language_consistent"])),
             ("citation hit (an expected document is cited)", *rate(gen.loc[gen.scenario_id != "OOS", "citation_hit"])),
             ("citation precision (cited documents that are expected)", *rate(gen.loc[gen.scenario_id != "OOS", "citation_precision"])),
             ("answers with server-removed unsupported figures", *rate((gen["removed_unsupported"] > 0).astype(float))),
             ("answers with raw [n] markers", *rate(gen["raw_markers"].astype(float))),
             ("answers with template endings", *rate(gen["template_endings"].map(bool).astype(float))),
             ("supporter offer on in-scope questions", *rate(gen.loc[gen.scenario_id != "OOS", "offer_handoff"].astype(float)))]
    auto = pd.DataFrame(rows, columns=["metric", "value", "n"]); display(auto.round(3))
    oos = answers[answers.scenario_id == "OOS"][["question_id", "answer_generation", "evidence_level", "answer"]]
    display(Markdown("**Out-of-scope questions (appropriate abstention, judged by reading):**")); display(oos)

In [ ]:
if len(answers):
    gen = answers[answers["answer_generation"] == "llm"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    oc = outcome.value_counts()
    axes[0].bar(oc.index.astype(str), oc.values, color=A.SINGLE, width=0.55, zorder=2)
    for x, v in enumerate(oc.values): axes[0].text(x, v + 0.2, str(v), ha="center", fontsize=9)
    axes[0].set_title(f"Generation outcome (n = {len(answers)})"); axes[0].set_ylabel("cases"); axes[0].tick_params(axis="x", labelsize=8)
    timed = answers["wall_latency_ms"].dropna()
    axes[1].hist(timed / 1000, bins=12, color=A.SINGLE, edgecolor=A.SURFACE, zorder=2)
    axes[1].set_title(f"End-to-end response time (n = {len(timed)} timed)"); axes[1].set_xlabel("seconds"); axes[1].set_ylabel("cases")
    if len(gen):
        axes[2].hist(gen["total_tokens"].dropna(), bins=10, color=A.SINGLE, edgecolor=A.SURFACE, zorder=2)
    axes[2].set_title(f"Tokens per generated answer (n = {gen['total_tokens'].notna().sum()})"); axes[2].set_xlabel("total tokens (API-reported)")
    fig.tight_layout(); print(A.save(fig, "05_answer_outcomes_latency_tokens"))
    display(answers.groupby("answer_generation", dropna=False)[["wall_latency_ms", "llm_latency_ms", "input_tokens", "output_tokens", "total_tokens"]].describe().T.round(0))

## Human-judged quality (rubric v1)

In [ ]:
scored = annotations.dropna(subset=[c for c in ("correctness", "relevance", "faithfulness") if c in annotations.columns], how="all") \
    if len(annotations) else annotations
if len(scored) == 0:
    display(Markdown("""**Not evaluated.** Answer correctness, answer relevance, faithfulness, citation accuracy (rubric) and the
unsupported-claim rate need human review (or a recorded LLM judge) against the approved sources. No reviewed scores exist
yet - `data/review/answer_annotation_template.csv` lists every recorded answer for scoring. No model-generated or simulated
scores are substituted."""))
else:
    display(scored.groupby("reviewer")[["correctness", "relevance", "faithfulness", "citation_accuracy"]].mean())